In [0]:
# ============================================================
# 07_DASHBOARD_ANALYTICS
# ============================================================

print("=" * 70)
print("TELECOM ANALYTICS DASHBOARD")
print("=" * 70)
print("Gold analytical layer connected successfully")
print("=" * 70)

spark.sql("USE CATALOG telecom")
spark.sql("USE SCHEMA gold")

print("✅ Catalog : telecom")
print("✅ Schema  : gold")

In [0]:
# ============================================================
# EXECUTIVE KPIs
# ============================================================

executive_kpis = spark.sql("""
SELECT *
FROM telecom.gold.vw_executive_kpis
""")

display(executive_kpis)

In [0]:
# ============================================================
# MONTHLY REVENUE
# ============================================================

monthly_revenue = spark.sql("""
SELECT
    year_month,
    total_bills,
    gross_billed_amount,
    total_tax,
    total_discounts,
    net_billed_amount,
    paid_bills,
    unpaid_bills,
    overdue_bills
FROM telecom.gold.vw_revenue_analysis
ORDER BY year, month
""")

display(monthly_revenue)

In [0]:
# ============================================================
# MONTHLY USAGE
# ============================================================

usage_analysis = spark.sql("""
SELECT
    year_month,
    total_calls,
    total_call_minutes,
    total_sms,
    total_data_gb,
    call_charges,
    sms_charges,
    data_charges,
    total_usage_charges
FROM telecom.gold.vw_usage_analysis
ORDER BY year, month
""")

display(usage_analysis)

In [0]:
# ============================================================
# PLAN PERFORMANCE
# ============================================================

plan_performance = spark.sql("""
SELECT
    plan_id,
    plan_name,
    plan_type,
    monthly_charge,
    total_subscriptions,
    active_subscriptions,
    cancelled_subscriptions,
    total_bills,
    total_revenue,
    total_calls,
    total_call_minutes,
    total_data_gb
FROM telecom.gold.vw_plan_performance
ORDER BY total_revenue DESC
""")

display(plan_performance)

In [0]:
# ============================================================
# COMPLAINT ANALYSIS
# ============================================================

complaint_analysis = spark.sql("""
SELECT
    category_name,
    total_complaints,
    open_complaints,
    in_progress_complaints,
    resolved_complaints,
    high_priority_complaints,
    critical_complaints,
    avg_resolution_days
FROM telecom.gold.vw_complaint_analysis
ORDER BY total_complaints DESC
""")

display(complaint_analysis)

In [0]:
# ============================================================
# CHURN RISK
# ============================================================

churn_risk = spark.sql("""
SELECT
    customer_id,
    customer_name,
    current_plan,
    customer_status,
    total_subscriptions,
    active_subscriptions,
    cancelled_subscriptions,
    outstanding_amount,
    total_complaints,
    high_priority_complaints,
    failed_payments,
    payment_success_rate,
    churn_risk_score,
    churn_risk_level
FROM telecom.gold.vw_churn_risk
ORDER BY
    churn_risk_score DESC,
    outstanding_amount DESC
""")

display(churn_risk)

In [0]:
# ============================================================
# CHURN RISK SUMMARY
# ============================================================

churn_risk_summary = spark.sql("""
SELECT
    churn_risk_level,
    COUNT(*) AS customer_count,
    ROUND(
        AVG(churn_risk_score),
        2
    ) AS average_risk_score,
    ROUND(
        SUM(outstanding_amount),
        2
    ) AS outstanding_amount
FROM telecom.gold.customer_360
GROUP BY churn_risk_level
ORDER BY
    CASE churn_risk_level
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
        WHEN 'LOW' THEN 3
    END
""")

display(churn_risk_summary)

In [0]:
# ============================================================
# TOP HIGH-RISK CUSTOMERS
# ============================================================

top_risk_customers = spark.sql("""
SELECT
    customer_id,
    customer_name,
    customer_status,
    current_plan,
    outstanding_amount,
    total_complaints,
    high_priority_complaints,
    failed_payments,
    payment_success_rate,
    churn_risk_score,
    churn_risk_level
FROM telecom.gold.customer_360
WHERE churn_risk_level = 'HIGH'
ORDER BY
    churn_risk_score DESC,
    outstanding_amount DESC
LIMIT 20
""")

display(top_risk_customers)

In [0]:
# ============================================================
# ADDITIONAL TELECOM KPI SUMMARY
# ============================================================

additional_kpis = spark.sql("""
SELECT

    (
        SELECT COUNT(*)
        FROM telecom.gold.dim_subscription
        WHERE activation_date IS NOT NULL
    ) AS new_subscribers,

    (
        SELECT COUNT(*)
        FROM telecom.gold.dim_subscription
        WHERE deactivation_date IS NOT NULL
    ) AS churned_subscribers,

    (
        SELECT ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN deactivation_date IS NOT NULL THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        )
        FROM telecom.gold.dim_subscription
    ) AS churn_rate_pct,

    (
        SELECT ROUND(
            AVG(
                (
                    unix_timestamp(call_end_time)
                    - unix_timestamp(call_start_time)
                ) / 60.0
            ),
            2
        )
        FROM telecom.gold.fact_call_usage
        WHERE call_start_time IS NOT NULL
          AND call_end_time IS NOT NULL
    ) AS average_call_duration_minutes,

    (
        SELECT ROUND(
            SUM(data_consumed_mb) / 1024.0
            / COUNT(DISTINCT customer_id),
            2
        )
        FROM telecom.gold.fact_data_usage
    ) AS average_data_usage_gb_per_customer,

    (
        SELECT ROUND(
            total_collected_revenue /
            NULLIF(total_customers, 0),
            2
        )
        FROM telecom.gold.vw_executive_kpis
    ) AS arpu,

    (
        SELECT ROUND(
            100.0 *
            SUM(successful_payment_count) /
            NULLIF(SUM(total_payments), 0),
            2
        )
        FROM telecom.gold.customer_360
    ) AS payment_success_rate_pct,

    (
        SELECT ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN cancelled_subscriptions = 0 THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        )
        FROM telecom.gold.customer_360
    ) AS customer_retention_rate_pct

""")

display(additional_kpis)

In [0]:
%sql
SELECT *
FROM telecom.gold.vw_additional_telecom_kpis;